# 추론 모델 밑바닥부터 3편 실습 노트북 — 4장: 추론 시간 확장 ① 샘플링과 다수결 (한국어 주석판)

**출처:** 원본 노트북 `ch04_main.ipynb`, Sebastian Raschka, https://github.com/rasbt/reasoning-from-scratch , Apache License 2.0. 책 *Build a Reasoning Model (From Scratch)*, Manning.

**변경 사항:** 코난쌤(@conanssam)이 설명 글을 한국어로 옮기고 코드에 한국어 주석을 덧붙였습니다. 코드 동작은 원본과 같습니다.

**쓰는 법**
1. 저장소를 내려받습니다: `git clone https://github.com/rasbt/reasoning-from-scratch.git`
2. 저장소 폴더에서 `uv sync`를 실행해 필요한 패키지를 설치합니다.
3. `uv run jupyter lab`으로 주피터랩을 열고, 이 파일을 `ch04/01_main-chapter-code/` 폴더에 넣어 실행합니다.

**팁:** 여러 번 뽑아 다수결하는 평가는 문항 수 × 뽑는 횟수만큼 오래 걸립니다. 처음에는 적은 문항으로 돌려 보세요.

<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://sebastianraschka.com">Sebastian Raschka</a>의 책 <a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a>에 딸린 실습 코드입니다<br>
<br>코드 저장소: <a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 4장: 추론 시간 확장(inference-time scaling)으로 추론 실력 높이기

이 노트북에서 쓰는 패키지입니다.

In [ ]:
from importlib.metadata import version

used_libraries = [
    "matplotlib",
    "reasoning_from_scratch",
    "torch",
    "tokenizers"  # Used by reasoning_from_scratch
]

# 설치된 패키지 버전을 하나씩 출력합니다
for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F01_raschka.webp" width=600>

&nbsp;
## 4.1 추론 시간 확장 알아보기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F02_raschka.webp" width=600>

- 위 그림은 OpenAI o1 블로그 글에서 아이디어를 얻었습니다. https://openai.com/index/learning-to-reason-with-llms/

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F03_raschka.webp" width=600>

- 세 방법 모두 MATH-500(앞 장에서 쓴 벤치마크)에서 기본 모델의 정확도를 두 배 넘게 끌어올립니다.
- 방법 1과 2는 이번 장에서 다룹니다. 방법 3은 다음 장에서 다룹니다.

&nbsp;
## 4.2 사전학습된 모델 불러오기

- 모델을 불러오는 코드는 2장, 3장과 비슷합니다.

In [ ]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

# 쓸 수 있는 가장 빠른 장치(GPU·MPS·CPU)를 고릅니다
device = get_device()

# Use CPU for the first run of this chapter
device = torch.device("cpu")

# 추론 학습을 하지 않은 기본(base) 모델과 토크나이저를 불러옵니다
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False
)

- 앞 장에서 다룬 MATH-500 데이터셋의 문제 하나로 모델을 시험해 봅시다.

In [ ]:
from reasoning_from_scratch.ch03 import render_prompt

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
# 문제를 3장과 같은 프롬프트 양식으로 감쌉니다
prompt = render_prompt(raw_prompt)

print(prompt)

- 아래 함수는 앞 장의 `generate_text_stream_concat`을 바탕으로 합니다. 나중에 다른 `generate_*` 함수를 넣을 수 있게 조금 고쳤습니다.

In [ ]:
from reasoning_from_scratch.ch02 import generate_text_basic_stream_cache


def generate_text_stream_concat_flex(
    model, tokenizer, prompt, device, max_new_tokens,
    verbose=False, 
    generate_func=None,  # New
    **generate_kwargs  # New
):

    # 생성 함수를 따로 주지 않으면 2장의 기본 함수를 씁니다
    if generate_func is None:  # New
        generate_func = generate_text_basic_stream_cache
        
    # 프롬프트를 토큰 ID로 바꾸고 배치 차원을 하나 붙입니다
    input_ids = torch.tensor(
        tokenizer.encode(prompt), device=device
        ).unsqueeze(0)

    generated_ids = []
    # 고른 생성 함수로 토큰을 하나씩 받아 모읍니다
    for token in generate_func(  # New
        model=model,
        token_ids=input_ids,
        max_new_tokens=max_new_tokens,
        eos_token_id=tokenizer.eos_token_id,
        **generate_kwargs,  # New
    ):
        next_token_id = token.squeeze(0)
        generated_ids.append(next_token_id.item())

        # verbose=True면 생성되는 토큰을 바로바로 화면에 찍습니다
        if verbose:
            print(
                tokenizer.decode(next_token_id.tolist()),
                end="",
                flush=True
            )
    # 모은 토큰 ID를 글자로 되돌려 돌려줍니다
    return tokenizer.decode(generated_ids)

In [ ]:
# 먼저 평소처럼(탐욕 디코딩) 답을 생성해 봅니다
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_basic_stream_cache  # NEW
)

- 참고로 정답은 83입니다.

&nbsp;
## 4.3 사고 사슬 프롬프트로 더 나은 답 만들기

- 모델 성능을 높이는 고전적인 방법이 있습니다. 프롬프트를 바꿔서 모델이 단계별로 생각하거나 설명하게 이끄는 방법입니다. 이를 사고 사슬(chain-of-thought) 프롬프트라고 합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F04_raschka.webp" width=600>

In [ ]:
# 프롬프트 끝에 "단계별로 설명하라"는 한 줄만 덧붙입니다
prompt_cot = prompt + " \n\nExplain step by step."

response_cot = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True,
)

&nbsp;
## 4.4 온도 조절로 출력 다양성 다루기

- 지금까지는 결정적 디코딩(deterministic decoding)을 썼습니다. 이 절에서는 확률적 디코딩(stochastic decoding)을 소개합니다.
- 먼저 온도(temperature) 조절부터 봅니다. 그다음 4.5절에서 top-p를 더해 무작위성과 일관성의 균형을 더 잘 맞춥니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F05_raschka.webp" width=600>

&nbsp;
### 4.4.1 다음 토큰을 고르는 과정 이해하기

In [ ]:
ex_prompt = "The capital of Germany is"

# 다음 토큰 딱 하나만 생성해 봅니다
response = generate_text_stream_concat_flex(
    model, tokenizer, ex_prompt, device,
    max_new_tokens=1, verbose=True
)

- 안에서는 이런 일이 일어납니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F06_raschka.webp" width=600>

In [ ]:
# 1) Convert input text into token IDs

# 토크나이저로 문장을 숫자(토큰 ID)로 바꿉니다
input_token_ids = torch.tensor(
    tokenizer.encode(ex_prompt), device=device
).unsqueeze(0)

print(input_token_ids)

In [ ]:
# 2) Get scores for next token

with torch.inference_mode():
    # 마지막 위치의 출력만 꺼내면 "다음 토큰" 점수가 됩니다
    next_token_logits = model(input_token_ids)[:, -1]
print(next_token_logits.shape)  # Shape: [1, vocab_size]

In [ ]:
# 3) Find vocabulary index with highest score

# 점수가 가장 높은 토큰 하나를 고릅니다(탐욕 디코딩)
max_token_id = torch.argmax(next_token_logits)
print(f"Token ID: {max_token_id}")
print(f"Decoded token: '{tokenizer.decode([max_token_id])}'")

- 이 LLM의 어휘(vocabulary)는 토큰 151,936개이고, 인덱스는 0부터 151,935까지입니다. 그림으로 보기 쉽게 인덱스 19,800~19,900 사이의 토큰 100개만 살펴봅니다.

In [ ]:
import matplotlib.pyplot as plt

# 어휘 일부 구간의 점수를 막대그래프로 그리는 함수입니다
def plot_scores_bar(
    next_token_logits, start=19_800, end=19_900,
    arrow=True, ylabel="Logit value"
):
    # Select vocabulary subsection
    x = torch.arange(start, end)

    # .cpu() is a shortcut for to(torch.device("cpu"))
    logits_section = next_token_logits[0, start:end].float().cpu()

    # Plot logits
    # 구간의 점수를 막대로 그립니다
    plt.bar(x, logits_section)
    plt.xlabel("Vocabulary index")
    plt.ylabel(ylabel)

    # Highlight max logit
    if arrow:
        # 가장 높은 막대에 "Berlin" 화살표를 붙입니다
        max_idx = torch.argmax(logits_section)
        plt.annotate(
            "Berlin",
            xy=(x[max_idx], logits_section[max_idx]),
            xytext=(x[max_idx] - 25, logits_section[max_idx] - 2),
            arrowprops={
                "facecolor": "black", "arrowstyle": "->", "lw": 1.5
            },
            fontsize=10,
        )

    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_scores_bar(next_token_logits)

&nbsp;
### 4.4.2 온도 매개변수로 토큰 점수(로짓) 다시 맞추기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F08_raschka.webp" width=600>

In [ ]:
def scale_logits_by_temperature(logits, temperature):
    if temperature <= 0:
        raise ValueError("Temperature must be positive")
    # 로짓을 온도로 나눕니다: 1보다 작으면 뾰족해지고, 크면 평평해집니다
    return logits / temperature

- 엄밀히는 막대그래프가 더 맞습니다. 하지만 선으로 그리면 차이가 더 잘 보입니다.

In [ ]:
def plot_logits_with_temperature(
    next_token_logits, start=19_800, end=19_900,
    temps=(0.5, 5.0),
):
    x = torch.arange(start, end)
    logits_orig = next_token_logits[0, start:end].float().cpu()

    # Apply temperature scaling
    # 온도 두 개(0.5, 5.0)로 각각 로짓을 나눠 봅니다
    logits_scaled = [
        scale_logits_by_temperature(logits_orig, T) for T in temps
    ]
    # Plot logits
    # 원래 로짓과 온도를 바꾼 로짓을 선으로 겹쳐 그립니다
    plt.plot(x, logits_orig, label="Original logits", lw=2)
    plt.plot(
        x, logits_scaled[0],
        label=f"T={temps[0]} (sharper)", ls="--", lw=1
    )
    plt.plot(
        x, logits_scaled[1],
        label=f"T={temps[1]} (flatter)", ls=":", lw=3
    )

    # Highlight max logit
    max_idx = torch.argmax(logits_orig)
    plt.annotate(
        "Berlin",
        xy=(x[max_idx], logits_orig[max_idx]),
        xytext=(x[max_idx] - 25, logits_orig[max_idx] + 2),
        arrowprops={"facecolor": "black", "arrowstyle": "->", "lw": 1.5},
        fontsize=12,
    )

    plt.xlabel("Vocabulary index")
    plt.ylabel("Logit value")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_logits_with_temperature(
    next_token_logits,
    temps=(0.5, 5.0)
)

&nbsp;
### 4.4.3 확률 분포에서 다음 토큰 뽑기

- 로짓(logit) 점수를 확률 점수로 바꿉니다. 확률 점수는 모두 더하면 1이 됩니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F10_raschka.webp" width=600>

In [ ]:
# Step 3.2: Rescale next-token scores
rescaled_logits = scale_logits_by_temperature(next_token_logits, 5.0)

# Step 3.3 Convert rescaled logits into probability scores
# 소프트맥스로 점수를 확률(합이 1)로 바꿉니다
next_token_probas = torch.softmax(
    rescaled_logits, dim=-1
)

print("Probability sum:", torch.sum(next_token_probas))

In [ ]:
plot_scores_bar(
    next_token_probas, arrow=False, ylabel="Probability value"
)

In [ ]:
print("Token ID 19,846 probability:", next_token_probas[:, 19846])

In [ ]:
print("Highest probability:", max(next_token_probas.squeeze(0)))

In [ ]:
# Step 3.4: Sample token according to probabilities
# 시드를 고정해서 매번 같은 결과가 나오게 합니다
torch.manual_seed(123)
print(
    "Sampled token:",
    # 확률에 비례해서 토큰 하나를 무작위로 뽑습니다
    torch.multinomial(next_token_probas.cpu(), num_samples=1)
)

In [ ]:
print(tokenizer.decode([65094]))

In [ ]:
def count_samples(probas, num_samples=1000, threshold=1, tokenizer=None):
    # Draw samples according to probabilities
    # 확률대로 여러 번(기본 1000번) 뽑습니다
    samples = torch.multinomial(
        probas.cpu(), num_samples=num_samples, replacement=True
    )
    
    # Count how often each index was selected
    # 토큰마다 몇 번 뽑혔는지 셉니다
    counts = torch.bincount(samples.squeeze(0), minlength=1)
    
    # Print results
    for i, c in enumerate(counts):
        # threshold보다 많이 뽑힌 토큰만 출력합니다
        if c > threshold:
            if tokenizer is None:
                print(f"Vocab index {i}: {c.item()}x")
            else:
                print(f"'{tokenizer.decode([i])}': {c.item()}x")

In [ ]:
torch.manual_seed(123)
# 온도 5.0일 때 1000번 뽑으면 어떤 토큰이 나오는지 셉니다
count_samples(next_token_probas, tokenizer=tokenizer)

In [ ]:
torch.manual_seed(123)
# 이번엔 낮은 온도(0.35)로 바꿔서 다시 셉니다
probas_lowT = torch.softmax(
    scale_logits_by_temperature(next_token_logits, 0.35), dim=-1
)
count_samples(probas_lowT, threshold=1, tokenizer=tokenizer)

In [ ]:
print(probas_lowT[0, 19_846])

- `"______"`가 나온 이유는 아마 이렇습니다. 모델이 학습 데이터에서 이런 서식을 본 적이 있어서, 사용자에게 퀴즈를 만들어 주려는 것 같습니다.

&nbsp;
### 4.4.4 글 생성 함수에 온도 조절 넣기

In [ ]:
# Exercise 2.2 Appendix B
from reasoning_from_scratch.qwen3 import KVCache

@torch.inference_mode()
def generate_text_temp_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
    temperature=0.
):
    model.eval()
    # KV 캐시를 새로 만들어 이전 계산을 재활용합니다
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    # Step 3.1: Get logits
    out = model(token_ids, cache=cache)[:, -1]
    for step in range(max_new_tokens):

        ########################################
        # NEW:
        orig_device = token_ids.device

        # 온도가 0이면 예전처럼 가장 높은 점수의 토큰을 고릅니다
        if temperature is None or temperature == 0.0:
            next_token = torch.argmax(out, dim=-1, keepdim=True)

        else:
            # Step 3.2: Apply temperature scaling on logits
            logits = scale_logits_by_temperature(out, temperature)

            # Step 3.3: Convert to probabilities
            probas = torch.softmax(logits, dim=-1)

            # Step 3.4: Sample token according to probabilities
            next_token = torch.multinomial(probas.cpu(), num_samples=1)
            next_token = next_token.to(orig_device)
            
        #########################################
        # 끝 토큰(EOS)이 나오면 생성을 멈춥니다
        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        # 토큰을 하나씩 바깥으로 내보냅니다
        yield next_token
        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            out = model(next_token, cache=cache)[:, -1]

In [ ]:
torch.manual_seed(123)
# 온도 1.1로 무작위성을 넣어 답을 생성합니다
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_temp_stream_cache,
    temperature=1.1
)

&nbsp;
## 4.5 top-p 샘플링으로 다양성과 일관성의 균형 맞추기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F12_raschka.webp" width=600>

&nbsp;
### 4.5.1 top-p 토큰 묶음 고르기

In [ ]:
# Step 3.1: Get logits (here: use toy logits for 10 tokens)
toy_logits = torch.tensor(
    [-0.7, -3.0, 0.1, -1.2, 2.0, -1.0, -0.5, -2.0, 0.3, 1.5]
)

# Step 3.2: Apply temperature scaling
toy_logits_scaled = scale_logits_by_temperature(toy_logits, 1.0)

# Step 3.3: Convert to probabilities
toy_probas = torch.softmax(toy_logits_scaled, dim=-1)

# 토큰 10개의 확률을 막대로 그립니다
plt.bar(
    torch.arange(len(toy_logits_scaled)), toy_probas,
    alpha=0.5
)

plt.ylim([0, 1])
plt.xlabel("Vocabulary index")
plt.ylabel("Probability")
# plt.savefig("12.pdf")
plt.show()

In [ ]:
# Step 4.1: Sort by descending probability
sorted_probas, sorted_idx = torch.sort(toy_probas, descending=True)

# Step 4.2: Compute cumulative sum
cumsum = torch.cumsum(sorted_probas, dim=-1)

plt.bar(
    torch.arange(len(sorted_probas)), sorted_probas, 
        alpha=0.5
)
# 누적합을 계단 선으로 겹쳐 그립니다
plt.step(
    torch.arange(len(cumsum)), cumsum, 
    where="mid", color="C1", label="Cumulative sum"
)

plt.ylim([0, 1])
plt.xlabel("Token rank (sorted by probability)")
plt.ylabel("Probability")
plt.show()

In [ ]:
# Step 4.3.1: Apply top-p threshold (e.g., keep tokens until cumulative mass > 0.8)
top_p = 0.8
# 누적합이 0.8 이하인 토큰만 남깁니다
keep_mask = cumsum <= top_p
n_kept = torch.sum(keep_mask).item()
print("Cumulative sum:", cumsum)
print("Tokens kept:", n_kept)

In [ ]:
# A more common variant is to include the token that crosses the threshold
# "이 토큰 앞까지의 누적합"으로 비교하면 경계를 넘는 토큰도 포함됩니다
keep_mask = (cumsum - sorted_probas) < top_p
n_kept = keep_mask.sum().item()
print("Tokens kept:", n_kept)

In [ ]:
plt.bar(
    torch.arange(len(sorted_probas)), sorted_probas, 
    alpha=0.5, label="Sorted probabilities"
)
plt.step(
    torch.arange(len(cumsum)), cumsum, where="mid",
    color="darkorange", label="Cumulative sum"
)

# Highlight cutoff
# 빨간 점선은 top_p 기준선, 회색 점선은 잘리는 위치입니다
plt.axhline(
    top_p, color="red", linestyle="--",
    label=f"top_p = {top_p}"
)
plt.axvline(
    n_kept - 0.5, color="gray", linestyle=":",
    label=f"Top-p cutoff at {n_kept} tokens"
)

plt.xlabel("Token rank (sorted by probability)")
plt.ylabel("Probability")
plt.legend()
plt.grid(alpha=0.3)
plt.ylim(0, 1.05)
# plt.savefig("14.pdf")
plt.show()

In [ ]:
# Step 4.3.2: Zero out beyond cutoff
# 남기지 않을 토큰의 확률은 0으로 만듭니다
kept_sorted = torch.where(
    keep_mask, sorted_probas,
    torch.zeros_like(sorted_probas)
)

# Step 4.3.3: Map back to original order
filtered = torch.zeros_like(toy_probas).scatter(0, sorted_idx, kept_sorted)

print(filtered)

In [ ]:
# Step 4.4: Renormalize to sum to 1
denom = torch.sum(filtered).clamp_min(1e-12)
# 남은 확률을 다시 합이 1이 되게 나눕니다
renormalized = filtered / denom
print(renormalized)

&nbsp;
### 4.5.2 글 생성 함수에 top-p 필터 넣기

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F16_raschka.webp" width=600>

In [ ]:
def top_p_filter(probas, top_p):
    # top_p가 없거나 1 이상이면 아무것도 거르지 않습니다
    if top_p is None or top_p >= 1.0:
        return probas

    # Step 4.1: Sort by descending probability
    sorted_probas, sorted_idx = torch.sort(probas, dim=1, descending=True)

    # Step 4.2: Cumulative sum
    cumprobas = torch.cumsum(sorted_probas, dim=1)

    # Step 4.3.1: Keep tokens where prefix cumulative mass (before token) is < top_p
    # Example: [0.5, 0.41, 0.09] with top_p=0.9 should keep the first two tokens
    prefix = cumprobas - sorted_probas   # cumulative mass before each token
    keep = prefix < top_p
    # Always keep at least one token (fallback for very small/non-positive top_p)
    keep[:, 0] = True    

    # Step 4.3.2: Zero out beyond cutoff
    kept_sorted = torch.where(
        keep, sorted_probas,
        torch.zeros_like(sorted_probas)
    )
    # Step 4.3.3: Map back to original order
    filtered = torch.zeros_like(probas).scatter(1, sorted_idx, kept_sorted)

    # Step 4.4: Renormalize to sum to 1
    denom = torch.sum(filtered, dim=1, keepdim=True).clamp_min(1e-12)
    # 남은 토큰의 확률만으로 다시 합이 1이 되게 합니다
    return filtered / denom

In [ ]:
with torch.inference_mode():
    # "The capital of Germany is" 다음 토큰 점수를 다시 구합니다
    next_token_logits = model(input_token_ids)[:, -1]
print(next_token_logits.shape)

In [ ]:
torch.manual_seed(123)
probas_lowT = torch.softmax(
    scale_logits_by_temperature(next_token_logits, 0.35), dim=-1
)
# 먼저 top-p 없이 낮은 온도로만 뽑아 봅니다
count_samples(probas_lowT, threshold=1, tokenizer=tokenizer)

In [ ]:
torch.manual_seed(123)
probas_lowT = torch.softmax(
    scale_logits_by_temperature(next_token_logits, 0.35), dim=-1
)
# top-p 0.8 필터를 거치면 엉뚱한 토큰이 빠집니다
probas_lowT_filtered = top_p_filter(probas_lowT, top_p=0.8)

count_samples(probas_lowT_filtered, threshold=1, tokenizer=tokenizer)

In [ ]:
@torch.inference_mode()
def generate_text_top_p_stream_cache(
    model,
    token_ids,
    max_new_tokens,
    eos_token_id=None,
    temperature=0.,
    top_p=None
):
    model.eval()
    cache = KVCache(n_layers=model.cfg["n_layers"])
    model.reset_kv_cache()

    # Step 3.1: Get logits
    out = model(token_ids, cache=cache)[:, -1]
    for step in range(max_new_tokens):

        orig_device = token_ids.device

        # 온도가 0이면 탐욕 디코딩, 아니면 온도 + top-p 샘플링입니다
        if temperature is None or temperature == 0.0:
            next_token = torch.argmax(out, dim=-1, keepdim=True)

        else:
            # Step 3.2: Apply temperature scaling on logits
            logits = scale_logits_by_temperature(out, temperature)

            # Step 3.3: Convert to probabilities
            probas = torch.softmax(logits, dim=-1)

            # (New) Step 4: Apply top-p filter to probabilities
            probas = top_p_filter(probas, top_p)            

            # Step 3.4: Sample token according to probabilities
            next_token = torch.multinomial(probas.cpu(), num_samples=1)
            next_token = next_token.to(orig_device)

        if (eos_token_id is not None
                and torch.all(next_token == eos_token_id)):
            break

        # 토큰을 하나씩 바깥으로 내보냅니다
        yield next_token
        if step + 1 < max_new_tokens:  # Skips the forward pass after the last token is generated
            out = model(next_token, cache=cache)[:, -1]

In [ ]:
torch.manual_seed(123)
# 온도 0.5 + top-p 0.8로 답을 생성합니다
response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.5,
    top_p=0.8, 
)

&nbsp;
## 4.6 자기 일관성으로 답의 정확도 높이기

- 샘플링으로 답(또는 추론 과정)을 여러 개 만들고 투표합니다.

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F17_raschka.webp" width=600>

- 고전적인 다수결 투표(majority voting)와 비슷합니다. 다만 서로 다른 모델 대신 서로 다른 온도 설정을 씁니다.
- 이 방법은 [Self-Consistency Improves Chain-of-Thought Reasoning in Language Models](https://arxiv.org/abs/2203.11171) 논문에도 나옵니다. 이름은 자기 일관성(self-consistency)입니다.
- 짧게 정리하면 이 방법은
  - 높은 온도와 top-p로 추론 과정을 여러 개 뽑고
  - 각 과정에서 최종 답을 꺼낸 뒤
  - 가장 자주 나온 최종 답을 고릅니다.
- 덧붙이면, 이건 엄밀히 말해 다수결이 아니라 [최다 득표(plurality vote)](https://en.wikipedia.org/wiki/Plurality_(voting))입니다(미국 영어 기준).

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F18_raschka.webp?1" width=600>

In [ ]:
from reasoning_from_scratch.ch03 import extract_final_candidate
from collections import Counter

def self_consistency_vote(
    model, tokenizer, prompt, device,
    num_samples=10, temperature=0.8, top_p=0.9, max_new_tokens=2048,
    show_progress=True, show_long_answer=False, seed=None,
):
    # 긴 답(풀이 전체)과 짧은 답(최종 답)을 따로 모읍니다
    full_answers, short_answers = [], []

    # 1) Sample multiple answers
    for i in range(num_samples):
        if seed is not None:
            # 샘플마다 시드를 다르게 줘서 서로 다른 답이 나오게 합니다
            torch.manual_seed(seed + i + 1)

        answer = generate_text_stream_concat_flex(
            model=model, tokenizer=tokenizer, prompt=prompt, device=device,
            max_new_tokens=max_new_tokens, verbose=show_long_answer,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature, top_p=top_p,
        )

        # 2) Extract the final (short) answer from each answer
        short = extract_final_candidate(
            answer, fallback="number_then_full"
        )
        full_answers.append(answer)
        short_answers.append(short)
        if show_progress:
            print(f"[Sample {i+1}/{num_samples}] → {short!r}")

    # 3) Choose the most frequent final answer (self-consistency vote)
    # 최종 답마다 몇 번 나왔는지 셉니다
    counts = Counter(short_answers)
    groups = {s: [] for s in counts}
    for idx, s in enumerate(short_answers):
        groups[s].append(idx)

    mc = counts.most_common()
    if not mc:
        majority_winners, final_answer = [], None
    else:
        top_freq = mc[0][1]
        # 가장 많이 나온 답이 여럿이면 동점이라 final_answer는 None이 됩니다
        majority_winners = [s for s, f in mc if f == top_freq]
        final_answer = mc[0][0] if len(majority_winners) == 1 else None

    return {
        "full_answers": full_answers,
        "short_answers": short_answers,
        "counts": dict(counts),
        "groups": groups,
        "majority_winners": majority_winners,
        "final_answer": final_answer,
    }

- 추가 팁
  - (긴) 답들이 거의 똑같아 보이면, 온도를 조금씩 올려서 다양성을 키우세요.
  - (긴) 답들이 이상해 보이면, 온도를 낮추세요.

In [ ]:
# 같은 문제를 5번 뽑아 가장 많이 나온 답을 고릅니다
results = self_consistency_vote(
    model,
    tokenizer,
    prompt,
    device=device,
    num_samples=5,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    seed=123,
    show_progress=True,
)

In [ ]:
print(results["final_answer"])

In [ ]:
print(results["full_answers"][0])

In [ ]:
# 이번엔 "단계별로 설명하라"(CoT)를 붙여서 투표합니다
results = self_consistency_vote(
    model,
    tokenizer,
    prompt + "\n\nExplain step by step.",
    device=device,
    num_samples=5,
    temperature=0.8,
    top_p=0.9,
    max_new_tokens=2048,
    seed=123,
    show_progress=True,
)

- 이런 병렬 샘플링은 Claude 4도 씁니다(https://www.anthropic.com/news/claude-4). 여기에 더해 Claude 4는 답에 점수를 매기는 내부 모델을 씁니다.
- 점수 매기는 모델은 부록 F의 LLM 심판(F.5 Judging responses with other LLMs)과 비슷하게 만들 수 있습니다.
- 다음 장에서는 신뢰도 점수(confidence score)로 점수를 매기는 방법도 소개합니다. 이 점수는 동점일 때 승부를 가르는 데도 쓸 수 있습니다.

아래는 cuda 장치(DGX Spark)에서 MATH-500 테스트셋 전체 500문제로 여러 방법을 평가한 결과입니다.

|    | 방법                                         | 모델      | 정확도   | 걸린 시간  |
|----|----------------------------------------------|-----------|----------|------------|
| 1  | 기준선(3장), 탐욕 디코딩(greedy decoding)    | Base      | 15.2%    | 10.1 min   |
| 2  | 기준선(3장), 탐욕 디코딩                     | Reasoning | 48.2%    | 182.1 min  |
| 3  | 사고 사슬 프롬프트("CoT")                    | Base      | 40.6%    | 84.5 min   |
| 4  | 온도와 top-p("Top-p")                        | Base      | 17.8%    | 30.7 min   |
| 5  | "Top-p" + 자기 일관성 (n=3)                  | Base      | 29.6%    | 97.6 min   |
| 6  | "Top-p" + 자기 일관성 (n=5)                  | Base      | 27.8%    | 116.8 min  |
| 7  | "Top-p" + 자기 일관성 (n=10)                 | Base      | 31.6%    | 300.4 min  |
| 8  | "Top-p" + "CoT"                              | Base      | 33.4%    | 129.2 min  |
| 9  | 자기 일관성 (n=3) + "Top-p" + "CoT"          | Base      | 42.2%    | 211.6 min  |
| 10 | 자기 일관성 (n=5) + "Top-p" + "CoT"          | Base      | 48.0%    | 452.9 min  |
| 11 | 자기 일관성 (n=10) + "Top-p" + "CoT"         | Base      | 52.0%    | 862.6 min  |
| 12 | 자기 일관성 (n=3) + "Top-p" + "CoT"          | Reasoning | 55.2%    | 544.4 min  |

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/ch04/CH04_F19_raschka.webp" width=600>

&nbsp;
## 4.7 정리

- 이 절에는 코드가 없습니다.